# 基于MindNLP 的 BERT 抽取式问答模型微调实战

## 1. 项目背景
机器阅读理解（Machine Reading Comprehension, MRC）是自然语言处理的核心任务之一。本实验旨在基于 **MindSpore 2.7.0** 深度学习框架与 **MindNLP 0.5.1** 自然语言处理套件，在经典的 **SQuAD (Stanford Question Answering Dataset)** 数据集上微调 BERT 模型，构建一个端到端的抽取式问答（Extractive QA）系统。

## 2. 实验目标与关键技术
* **模型架构**：使用 BERT-Base 模型作为骨干网络，通过下游任务微调实现答案跨度（Answer Span）预测。
* **长文本处理**：实现**滑动窗口（Sliding Window）**机制，有效解决 BERT 输入序列长度限制（512 Tokens）问题。
* **异构计算适配**：解决 MindSpore 在动态图模式（PyNative）下的张量设备同步（Device Mismatch）问题。
* **性能评估**：通过验证集样本进行定性分析，验证模型在真实场景下的推理能力。

## 3. 环境依赖
* Hardware: Ascend 910 / GPU (CUDA)
* Framework: MindSpore >= 2.7.0
* Toolkit: MindNLP == 0.5.1


In [ ]:
# 安装实验所需的依赖库
# 注意：首次运行需取消注释并执行,若mindnlp安装后报错可直接下载源码并执行 pip install -e .
# !pip install mindspore==2.7.0
# !pip install mindnlp==0.5.1
# !pip install datasets tqdm nbformat
# !diffusers==0.35.2

In [ ]:
import os
import numpy as np
import mindspore as ms
from mindnlp.transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer
from datasets import load_dataset 

# --- 全局环境配置 ---
# 自动检测计算设备：优先使用 Ascend NPU，其次尝试 CCPU
try:
    ms.set_context(device_target="Ascend")
except Exception:
    ms.set_context(device_target="CPU")

# 设置运行模式：推荐使用 PYNATIVE_MODE (动态图模式) 以获得更好的调试体验
ms.set_context(mode=ms.PYNATIVE_MODE)

print(f"当前运行设备 (Device): {ms.get_context('device_target')}")
print(f"当前运行模式 (Mode): {ms.get_context('mode')}")

## 4. 数据加载与预处理

本实验使用 Hugging Face 的 `datasets` 库加载 SQuAD 数据集。该数据集包含 问题 (Question)、上下文 (Context) 以及 答案 (Answer)。

In [ ]:
# 加载 SQuAD 数据集
print("正在加载数据集 (SQuAD)...")
squad_dataset = load_dataset("squad")

# === 实验数据采样 ===
# 为了演示流程的高效性，此处仅抽取部分数据进行训练和验证
# 在正式全量训练时，请使用完整数据集
train_dataset = squad_dataset["train"].select(range(5000))      # 训练集采样: 5000条
eval_dataset = squad_dataset["validation"].select(range(1500))   # 验证集采样: 1500条

print(f"训练集样本数: {len(train_dataset)}")
print(f"验证集样本数: {len(eval_dataset)}")
print(f"样本示例: {train_dataset[0]}")

### 4.1 数据预处理：Tokenization 与 滑动窗口

由于 SQuAD 中的部分文章长度超过 BERT 的最大输入限制（通常为 512 或 384），我们需要引入**滑动窗口（Sliding Window）**策略：
1.  **截断与步长**：当文本过长时，将其切分为多个包含重叠片段的特征（Features）。
2.  **标签对齐**：计算答案在切分后的每个特征片段中的新起始位置（Start/End Position）。


In [ ]:
# 模型与预处理超参数配置
model_checkpoint = "bert-base-uncased"
batch_size = 16
max_length = 384    # 输入序列最大长度
doc_stride = 128    # 滑动窗口步长 (重叠部分的长度)

# 初始化 Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def prepare_train_features(examples):
    '''
    数据预处理核心函数：
    1. 对 Question 和 Context 进行编码。
    2. 处理长文本溢出 (Overflow)，应用滑动窗口。
    3. 将字符级别的答案位置 (Character Index) 映射为 Token 级别的索引。
    '''
    # 去除问题两端的空白字符
    questions = [q.strip() for q in examples["question"]]
    
    # Tokenize
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",       # 仅截断 Context，保留 Question
        stride=doc_stride,              # 应用滑动窗口
        return_overflowing_tokens=True, # 允许返回多个片段
        return_offsets_mapping=True,    # 返回字符偏移量映射，用于定位答案
        padding="max_length"
    )

    # 获取映射关系
    sample_map = inputs.pop("overflow_to_sample_mapping")
    offset_mapping = inputs.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answers = examples["answers"][sample_idx]
        
        # 异常处理：如果样本没有答案，标注为 CLS (0, 0)
        if len(answers["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue
            
        # 获取答案在原文中的字符级起止位置
        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        # 区分 Sequence 中的 Question 部分和 Context 部分
        sequence_ids = inputs.sequence_ids(i)
        
        # 寻找 Context 的起止 Token 索引
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # 判断：如果答案并没有完全包含在当前的窗口片段中，则标记为 (0, 0)
        if not (offsets[context_start][0] <= start_char and offsets[context_end][1] >= end_char):
            start_positions.append(0)
            end_positions.append(0)
        else:
            # 否则，寻找答案 token 的起止索引
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

print("正在执行数据预处理 (Map)...")
tokenized_train = train_dataset.map(prepare_train_features, batched=True, remove_columns=train_dataset.column_names)
tokenized_eval = eval_dataset.map(prepare_train_features, batched=True, remove_columns=eval_dataset.column_names)
print("预处理完成！")

## 5. 模型微调 (Model Fine-tuning)

加载预训练的 BERT 模型，配置训练参数，并使用 MindNLP 的 `Trainer` 接口启动训练流程。

In [ ]:
# 加载预训练模型 (AutoModelForQuestionAnswering)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

# 配置训练参数 (TrainingArguments)
args = TrainingArguments(
    output_dir="./bert_qa_output",
    eval_strategy="no",             # 演示阶段不进行频繁评估以加速
    learning_rate=2e-5,             # 学习率
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,             # 训练轮次
    weight_decay=0.01,
    save_strategy="no",             # 不保存 Checkpoint
    logging_steps=10
)

# 初始化 Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
)

# 启动训练
print(">>> 开始模型训练...")
trainer.train()
print(">>> 模型训练结束。")

## 6. 模型推理与应用 (Inference)

为了验证模型效果，我们定义一个端到端的预测函数。

**技术难点说明**：
在使用 MindNLP 与 MindSpore 进行交互时，需特别注意**张量设备同步 (Device Synchronization)**。输入模型的 Tensor 必须显式移动到与模型相同的计算设备（Ascend/GPU）上，否则会引发 `ValueError: All tensor arguments must be on the same device`。

In [ ]:
import mindspore as ms
import numpy as np

def predict_answer(question, context):
    '''
    端到端预测函数：输入问题和上下文，输出预测的答案文本。
    包含设备自动适配逻辑。
    '''
    # 1. 切换模型至评估模式 (Evaluation Mode)
    model.set_train(False)
    
    # 2. 输入编码
    inputs = tokenizer(
        question, 
        context, 
        return_tensors="ms", 
        max_length=max_length, 
        truncation="only_second"
    )
    
    # 3. 设备同步 (Critical Step)
    # 获取模型当前所在的设备 (Ascend/CPU)
    try:
        target_device = model.device
    except:
        # 兼容性处理：如果无法直接获取，尝试通过参数或上下文推断
        try:
            target_device = next(model.get_parameters()).device
        except:
            target_device = ms.get_context("device_target")

    # 将输入张量移动到目标设备
    input_ids = ms.Tensor(inputs["input_ids"].asnumpy(), dtype=ms.int32).to(target_device)
    attention_mask = ms.Tensor(inputs["attention_mask"].asnumpy(), dtype=ms.int32).to(target_device)
    token_type_ids = ms.Tensor(inputs["token_type_ids"].asnumpy(), dtype=ms.int32).to(target_device)
    
    # 4. 前向传播 (Forward Pass)
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
    
    # 5. 结果解析 (Logits -> Text)
    start_logits = outputs.start_logits
    end_logits = outputs.end_logits
    
    # 获取概率最大的起始和结束位置索引
    start_idx = np.argmax(start_logits.asnumpy(), axis=-1)[0]
    end_idx = np.argmax(end_logits.asnumpy(), axis=-1)[0]
    
    # 简单的逻辑校验：如果结束位置在起始位置之前，做简单的回退处理
    if end_idx < start_idx:
        end_idx = start_idx + 10 
        
    # 解码答案
    answer_ids = inputs["input_ids"].asnumpy()[0][start_idx : end_idx + 1]
    answer = tokenizer.decode(answer_ids, skip_special_tokens=True)
    return answer

# 单例测试
print("--- 单例推理测试 ---")
context_demo = "MindSpore is a new open source deep learning training/inference framework that could be used for mobile, edge and cloud scenarios."
question_demo = "What is MindSpore?"
print(f"Q: {question_demo}")
print(f"A: {predict_answer(question_demo, context_demo)}")

## 7. 验证集效果抽样评估

从验证集（Validation Set）中随机抽取样本，对比“模型预测答案”与“数据集标准答案”，以定性评估模型性能。

In [ ]:
# 配置测试样本数
num_samples = 3
print(f"=== 正在从验证集抽取 {num_samples} 个样本进行评估 ===\n")

# 遍历验证集的前 N 个样本
# 注意：直接使用 Hugging Face Dataset 的索引访问，避免使用 create_dict_iterator
for i in range(num_samples):
    batch = eval_dataset[i]
    
    # 提取原始文本
    # Hugging Face Dataset 加载的数据默认为 Python str 类型
    question = batch['question']
    context = batch['context']
    
    # 解析参考答案 (SQuAD 格式较为复杂，此处提取第一个标准答案文本用于展示)
    try:
        ref_text_list = batch['answers']['text']
        ref_display = ref_text_list[0] if len(ref_text_list) > 0 else "<无答案>"
    except Exception as e:
        ref_display = f"答案解析错误: {str(e)}"

    # 执行模型预测
    pred_answer = predict_answer(question, context)
    
    # 打印对比结果
    print(f"【样本 Case {i+1}】")
    print(f"📌 问题: {question}")
    # 截断过长的上下文以便展示
    print(f"📄 原文: {context[:80]}...") 
    print(f"--------------------------------------------------")
    print(f"🟢 标准答案: {ref_display}")
    print(f"🔴 模型预测: {pred_answer}")
    print(f"==================================================\n")